# Car Model Recognition — Google Colab Training
BIL462 | Group: Junkers

**Gereksinimler:**
- Runtime → Change runtime type → **GPU (T4)**
- Kaggle API key (`kaggle.json`) — dataset indirmek için

In [ ]:
# Google Drive'ı bağla (checkpoint kaydetmek için)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ann-project'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print('Drive mounted:', DRIVE_DIR)

In [ ]:
# Kaggle API key yükle
from google.colab import files
print('kaggle.json dosyasını seç:')
uploaded = files.upload()

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle API key ayarlandı')

In [ ]:
# Dependency'ler ve repo
import subprocess, sys

subprocess.run(['pip', 'install', '-q', 'timm', 'kaggle'], check=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/umutgokmen/ann-project', '/content/ann-project'], check=True)
sys.path.insert(0, '/content/ann-project')
print('Setup complete')

In [ ]:
# Dataset indir (Drive'da zaten varsa bu hücreyi atla)
DATASET_DIR = f'{DRIVE_DIR}/stanford_cars'

if not os.path.exists(DATASET_DIR):
    print('Dataset indiriliyor...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'rickyyyyyyy/torchvision-stanford-cars',
        '-p', DATASET_DIR, '--unzip'
    ], check=True)
    print('Dataset indirildi:', DATASET_DIR)
else:
    print('Dataset zaten mevcut:', DATASET_DIR)

print('Içerik:', os.listdir(DATASET_DIR))

In [ ]:
# Dataset path'ini tespit et
from pathlib import Path

dataset_root = Path(DATASET_DIR)
# Kaggle dataset stanford_cars altında olabilir
if (dataset_root / 'stanford_cars').exists():
    dataset_root = dataset_root / 'stanford_cars'

print('Dataset root:', dataset_root)
print('Içerik:', [p.name for p in dataset_root.iterdir()])

In [ ]:
import yaml

with open('/content/ann-project/configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_dir']       = str(dataset_root)
cfg['data']['num_workers']       = 2
cfg['logging']['checkpoint_dir'] = f'{DRIVE_DIR}/checkpoints'
cfg['logging']['log_dir']        = f'{DRIVE_DIR}/logs'
cfg['training']['batch_size']    = 32
cfg['training']['epochs']        = 40
cfg['training']['amp']           = True

COLAB_CONFIG = '/content/config_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print(f"dataset_dir : {cfg['data']['dataset_dir']}")
print(f"checkpoints : {cfg['logging']['checkpoint_dir']}")
print(f"batch_size  : {cfg['training']['batch_size']}")
print(f"epochs      : {cfg['training']['epochs']}")
print(f"backbone    : {cfg['model']['backbone']}")

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
os.chdir('/content/ann-project')
from src.dataset import build_dataloaders
from src.model import build_model

loaders = build_dataloaders(cfg)
print(f"Train : {len(loaders['train'])} batch")
print(f"Val   : {len(loaders['val'])} batch")
print(f"Test  : {len(loaders['test'])} batch")
print(f"Class : {len(loaders['class_names'])}")

In [ ]:
# Sanity check — bir batch forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg).to(device)

images, labels = next(iter(loaders['train']))
with torch.no_grad():
    out = model(images.to(device))
print(f'Forward OK | input: {images.shape} | output: {out.shape}')

In [ ]:
# Training — ~2-3 saat T4'te
result = subprocess.run(
    ['python', 'src/train.py', '--config', COLAB_CONFIG],
    cwd='/content/ann-project'
)
print('Training exit code:', result.returncode)

In [ ]:
# Evaluation
result = subprocess.run([
    'python', 'src/evaluate.py',
    '--config', COLAB_CONFIG,
    '--checkpoint', f'{DRIVE_DIR}/checkpoints/best.pt',
    '--output_dir', f'{DRIVE_DIR}/evaluation',
], cwd='/content/ann-project')
print('Evaluation exit code:', result.returncode)

In [ ]:
from IPython.display import Image as IPImage
IPImage(f'{DRIVE_DIR}/evaluation/confusion_matrix.png')